# KMM-Scandaten Exploration & Preprocessing

Exploration realer Scandaten vom Koordinatenmessgerät (Hexagon / PC-DMIS).  
Dateiformat: `.xyz` mit erweitertem Header (Scanlinien-Normalen).

**Ziele:**
1. Laden und Visualisieren der rohen Punktwolke
2. Analyse der Scanlinien-Metadaten (Richtung, Dichte)
3. Preprocessing-Steps einzeln anwenden und bewerten
4. Vergleich Roh vs. Preprocessed

**Verwendung:** `SCAN_FILE` in Zelle 2 anpassen, dann alle Zellen ausführen.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd

project_root = Path().resolve().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from schweiss_ki.preprocessing import (
    PreprocessingPipeline,
    StatisticalOutlierFilter,
    RadiusOutlierFilter,
    VoxelGridDownsampler,
    NormalEstimator,
)
from schweiss_ki.core.data_structures import WeldVolumeModel

print('Imports OK')

## 1 – Konfiguration

In [ ]:
SCAN_FILE      = Path('../data/raw/cmm_scans/Schweißspalt_1,0_auf_2,5.xyz')
VIZ_MAX_POINTS = 50_000

print(f'Scan-Datei: {SCAN_FILE}')
print(f'Existiert:  {SCAN_FILE.exists()}')

## 2 – XYZ Laden & Header parsen

Das PC-DMIS `.xyz`-Format hat Scanlinien-Header im Format:
```
L{nr}##{color1}##{color2}##{nx}##{ny}##{nz}
```
Die Datenpunkte haben 7 Spalten: `x, y, z, vx, vy, vz, col7` (Bedeutung col7 tbd).

Open3D liest die ersten 3 Spalten (XYZ) direkt – die Header werden übersprungen.

In [ ]:
# --- Open3D: direktes Laden (nur XYZ) ---
pcd_raw = o3d.io.read_point_cloud(str(SCAN_FILE))
pts_raw = np.asarray(pcd_raw.points)

bbox = pcd_raw.get_axis_aligned_bounding_box()
extent = bbox.get_extent()

print(f'Punkte:       {len(pcd_raw.points):,}')
print(f'Bounding Box: {extent[0]:.1f} x {extent[1]:.1f} x {extent[2]:.1f} mm')
print(f'Hat Normalen: {pcd_raw.has_normals()}')
print(f'Hat Farben:   {pcd_raw.has_colors()}')

In [ ]:
# --- Erweiterter Parser: Header-Metadaten + alle 7 Spalten ---
def parse_cmm_xyz(filepath: Path) -> dict:
    """
    Parst PC-DMIS .xyz mit Scanlinien-Headern.
    
    Returns:
        dict mit:
        - 'points': np.ndarray (N, 3) – XYZ Koordinaten
        - 'all_columns': np.ndarray (N, 7) – alle Spalten
        - 'scan_line_id': np.ndarray (N,) – Scanlinien-Zuordnung pro Punkt
        - 'scan_normals': np.ndarray (N, 3) – Scannormale pro Punkt (vom Header)
        - 'headers': list[dict] – geparste Header-Infos
    """
    points = []
    all_columns = []
    scan_line_ids = []
    scan_normals = []
    headers = []
    
    current_line_id = None
    current_normal = None
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            if line.startswith('L'):
                parts = line.split('##')
                current_line_id = parts[0]  # z.B. 'L2'
                current_normal = [float(parts[3]), float(parts[4]), float(parts[5])]
                headers.append({
                    'line_id': current_line_id,
                    'color1': int(parts[1]),
                    'color2': int(parts[2]),
                    'normal': current_normal,
                })
            else:
                vals = line.split()
                if len(vals) >= 3:
                    row = [float(v) for v in vals]
                    points.append(row[:3])
                    all_columns.append(row)
                    scan_line_ids.append(current_line_id)
                    scan_normals.append(current_normal if current_normal else [0,0,0])
    
    return {
        'points': np.array(points),
        'all_columns': np.array(all_columns),
        'scan_line_id': np.array(scan_line_ids),
        'scan_normals': np.array(scan_normals),
        'headers': headers,
    }

parsed = parse_cmm_xyz(SCAN_FILE)

print(f'Punkte:         {parsed["points"].shape}')
print(f'Alle Spalten:   {parsed["all_columns"].shape}')
print(f'Scanlinien:     {len(parsed["headers"]):,}')

# Unique Scan-Richtungen
unique_normals = np.unique(parsed['scan_normals'], axis=0)
print(f'\nScan-Richtungen ({len(unique_normals)}):')
for n in unique_normals:
    mask = np.all(np.isclose(parsed['scan_normals'], n), axis=1)
    print(f'  n=[{n[0]:+.4f}, {n[1]:+.4f}, {n[2]:+.4f}]  →  {mask.sum():,} Punkte')

## 3 – Hilfsfunktionen

In [ ]:
def subsample(pcd, max_points=VIZ_MAX_POINTS):
    """Subsampled Punktwolke für Plotly-Visualisierung."""
    pts = np.asarray(pcd.points)
    if len(pts) > max_points:
        idx = np.random.default_rng(42).choice(len(pts), max_points, replace=False)
        pts = pts[idx]
    return pts


def plot_single(pcd, title, color='steelblue'):
    """3D-Scatter einer einzelnen Punktwolke."""
    pts = subsample(pcd)
    fig = go.Figure(go.Scatter3d(
        x=pts[:,0], y=pts[:,1], z=pts[:,2],
        mode='markers',
        marker=dict(size=1.5, color=color, opacity=0.7),
        name=f'{len(pcd.points):,} Punkte',
    ))
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
            aspectmode='data',
        ),
        height=600,
    )
    fig.show()


def plot_before_after(pcd_before, pcd_after, title,
                      color_before='steelblue', color_after='tomato'):
    """3D-Scatter Vergleich vorher/nachher."""
    pts_b = subsample(pcd_before)
    pts_a = subsample(pcd_after)
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=pts_b[:,0], y=pts_b[:,1], z=pts_b[:,2],
        mode='markers', marker=dict(size=1, color=color_before, opacity=0.3),
        name=f'Vorher: {len(pcd_before.points):,}',
    ))
    fig.add_trace(go.Scatter3d(
        x=pts_a[:,0], y=pts_a[:,1], z=pts_a[:,2],
        mode='markers', marker=dict(size=1.5, color=color_after, opacity=0.7),
        name=f'Nachher: {len(pcd_after.points):,}',
    ))
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
            aspectmode='data',
        ),
        height=600,
    )
    fig.show()

## 4 – Rohe Punktwolke

In [ ]:
plot_single(pcd_raw, f'Roh: {SCAN_FILE.stem} ({len(pcd_raw.points):,} Punkte)')

### 4.1 – Scan-Richtungen visualisieren

Die KMM hat aus 2 Richtungen (~±30° zur Vertikalen) gescannt, um die V-Naht abzubilden.

In [ ]:
# Punkte nach Scanrichtung einfärben
normals = parsed['scan_normals']
direction_label = np.where(normals[:, 0] < 0, 'Scan A (n_x < 0)', 'Scan B (n_x > 0)')

pts = parsed['points']
fig = go.Figure()
for label, color in [('Scan A (n_x < 0)', '#2196F3'), ('Scan B (n_x > 0)', '#FF5722')]:
    mask = direction_label == label
    fig.add_trace(go.Scatter3d(
        x=pts[mask, 0], y=pts[mask, 1], z=pts[mask, 2],
        mode='markers',
        marker=dict(size=1.2, color=color, opacity=0.6),
        name=f'{label}: {mask.sum():,} Punkte',
    ))
fig.update_layout(
    title='Scan-Richtungen (2 Winkel, ~±30° zur Vertikalen)',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data',
    ),
    height=600,
)
fig.show()

### 4.2 – Querschnitte (Y-Slices)

In [ ]:
y_min, y_max = pts[:, 1].min(), pts[:, 1].max()
y_positions = np.linspace(y_min + 5, y_max - 5, 6)
SLICE_THICKNESS = 0.5  # mm (±)

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'Y ≈ {y:.0f} mm' for y in y_positions],
)

for i, y in enumerate(y_positions):
    row = i // 3 + 1
    col = i % 3 + 1
    mask = np.abs(pts[:, 1] - y) < SLICE_THICKNESS
    sl = pts[mask]
    fig.add_trace(
        go.Scatter(
            x=sl[:, 0], y=sl[:, 2],
            mode='markers',
            marker=dict(size=2, color='steelblue'),
            name=f'{mask.sum()} pts',
            showlegend=False,
        ),
        row=row, col=col,
    )
    fig.update_xaxes(title_text='X (mm)', row=row, col=col)
    fig.update_yaxes(title_text='Z (mm)', row=row, col=col)

fig.update_layout(
    title=f'Querschnitte (XZ) bei verschiedenen Y-Positionen (±{SLICE_THICKNESS} mm)',
    height=700,
)
fig.show()

### 4.3 – Höhenprofil (Z-Verteilung)

In [ ]:
fig = px.histogram(
    x=pts[:, 2], nbins=150,
    title='Z-Wert-Verteilung (Höhenprofil)',
    labels={'x': 'Z (mm)', 'y': 'Anzahl'},
    color_discrete_sequence=['steelblue'],
)
fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Z=0')
fig.update_layout(height=400)
fig.show()

### 4.4 – Punktdichte pro Scanlinie

In [ ]:
line_ids, counts = np.unique(parsed['scan_line_id'], return_counts=True)

fig = px.histogram(
    x=counts, nbins=50,
    title=f'Punkte pro Scanlinie ({len(line_ids):,} Linien)',
    labels={'x': 'Punkte pro Linie', 'y': 'Anzahl Linien'},
    color_discrete_sequence=['steelblue'],
)
fig.update_layout(height=400)
fig.show()

print(f'Min: {counts.min()}, Max: {counts.max()}, Median: {np.median(counts):.0f}, Mean: {counts.mean():.1f}')

---
## 5 – Preprocessing Step-für-Step

Verwendet die bestehenden Preprocessing-Klassen aus `schweiss_ki.preprocessing`.  
Parameter hier zunächst mit Default-Werten — Tuning nach visueller Inspektion.

### 5.1 – Statistical Outlier Filter

In [ ]:
NB_NEIGHBORS = 20
STD_RATIO    = 2.0

f = StatisticalOutlierFilter(nb_neighbors=NB_NEIGHBORS, std_ratio=STD_RATIO)
pcd_stat, report_stat = f.apply(pcd_raw)

print(f'Vorher:   {report_stat.points_before:,} Punkte')
print(f'Nachher:  {report_stat.points_after:,} Punkte')
print(f'Entfernt: {report_stat.points_removed:,} ({100 - report_stat.retention_pct:.1f}%)')
print(f'Zeit:     {report_stat.duration_ms:.1f} ms')

plot_before_after(pcd_raw, pcd_stat,
    f'Statistical Outlier Filter (nb_neighbors={NB_NEIGHBORS}, std_ratio={STD_RATIO})')

### 5.2 – Radius Outlier Filter

Entfernt isolierte Punkte (z.B. Spritzer). Bei KMM-Daten evtl. weniger relevant als bei Laser-Scans.

In [ ]:
SEARCH_RADIUS = 1.5  # mm
MIN_NB_POINTS = 5

r = RadiusOutlierFilter(search_radius=SEARCH_RADIUS, min_nb_points=MIN_NB_POINTS)
pcd_radius, report_radius = r.apply(pcd_stat)  # auf bereits stat-gefilterte Daten

print(f'Vorher:   {report_radius.points_before:,} Punkte')
print(f'Nachher:  {report_radius.points_after:,} Punkte')
print(f'Entfernt: {report_radius.points_removed:,} ({100 - report_radius.retention_pct:.1f}%)')
print(f'Zeit:     {report_radius.duration_ms:.1f} ms')

plot_before_after(pcd_stat, pcd_radius,
    f'Radius Outlier Filter (radius={SEARCH_RADIUS}mm, min_pts={MIN_NB_POINTS})')

### 5.3 – Voxel Grid Downsampling

In [ ]:
VOXEL_SIZE = 0.5  # mm

d = VoxelGridDownsampler(voxel_size=VOXEL_SIZE)
pcd_voxel, report_voxel = d.apply(pcd_radius)

print(f'Vorher:   {report_voxel.points_before:,} Punkte')
print(f'Nachher:  {report_voxel.points_after:,} Punkte')
print(f'Entfernt: {report_voxel.points_removed:,} ({100 - report_voxel.retention_pct:.1f}%)')
print(f'Zeit:     {report_voxel.duration_ms:.1f} ms')

plot_before_after(pcd_radius, pcd_voxel,
    f'Voxel Grid Downsampling (voxel_size={VOXEL_SIZE}mm)')

### 5.4 – Normalenschätzung

Für RANSAC-Segmentierung (AP2.1 Phase 3) werden Normalen benötigt.  
Hier schauen wir, ob die Schätzung bei der KMM-Punktdichte sinnvoll funktioniert.

In [ ]:
NORMAL_RADIUS = 2.0  # mm

n = NormalEstimator(radius=NORMAL_RADIUS, orient_mode='consistent')
pcd_normals, report_normals = n.apply(pcd_voxel)

print(f'Normalen berechnet: {pcd_normals.has_normals()}')
print(f'Zeit:               {report_normals.duration_ms:.1f} ms')

# Normalen als Farbe darstellen (z-Komponente)
if pcd_normals.has_normals():
    normals_arr = np.asarray(pcd_normals.normals)
    pts_n = np.asarray(pcd_normals.points)
    nz = normals_arr[:, 2]  # z-Komponente der Normalen
    
    fig = go.Figure(go.Scatter3d(
        x=pts_n[:, 0], y=pts_n[:, 1], z=pts_n[:, 2],
        mode='markers',
        marker=dict(size=1.5, color=nz, colorscale='RdBu', cmin=-1, cmax=1,
                    colorbar=dict(title='Normal Z')),
        name=f'{len(pts_n):,} Punkte',
    ))
    fig.update_layout(
        title=f'Geschätzte Normalen (Z-Komponente), radius={NORMAL_RADIUS}mm',
        scene=dict(
            xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
            aspectmode='data',
        ),
        height=600,
    )
    fig.show()

---
## 6 – Komplette Pipeline

Alle Steps zusammen als `PreprocessingPipeline`, analog zu `pipeline.yaml`.

In [ ]:
pipeline = PreprocessingPipeline(
    steps=[
        StatisticalOutlierFilter(nb_neighbors=20, std_ratio=2.0),
        RadiusOutlierFilter(search_radius=1.5, min_nb_points=5),
        VoxelGridDownsampler(voxel_size=0.5),
        NormalEstimator(radius=2.0, orient_mode='consistent'),
    ],
    source_type='real',
)
print(f'Pipeline: {pipeline}')

pcd_final, report = pipeline.process(pcd_raw)

print(f'\nReport:')
print(f'  Punkte rein:  {report.points_in:,}')
print(f'  Punkte raus:  {report.points_out:,}')
print(f'  Retention:    {report.total_retention_rate:.1%}')
print(f'  Gesamt-Zeit:  {report.total_duration_ms:.0f} ms')
print()
for s in report.steps:
    print(f'  {s.step_name:<35} {s.points_before:>8,} → {s.points_after:>8,}  '
          f'(−{s.points_removed:,}, {s.duration_ms:.1f}ms)')

In [ ]:
plot_before_after(pcd_raw, pcd_final,
    'Komplette Pipeline: Roh vs. Preprocessed',
    color_before='steelblue', color_after='tomato')

## 7 – Report als Tabelle und Balkendiagramm

In [ ]:
rows = [{'Step': 'roh (input)', 'Punkte': report.points_in, 'Entfernt': 0,
         'Retention': '100%', 'Zeit (ms)': 0}]
for s in report.steps:
    rows.append({
        'Step':       s.step_name.replace('_', ' '),
        'Punkte':     s.points_after,
        'Entfernt':   s.points_removed,
        'Retention':  f'{s.retention_rate:.1%}',
        'Zeit (ms)':  round(s.duration_ms, 1),
    })

df = pd.DataFrame(rows)
display(df)

fig = px.bar(
    df, x='Step', y='Punkte',
    title='Punktanzahl nach jedem Preprocessing-Step',
    color='Punkte', color_continuous_scale='Blues',
    text='Retention',
)
fig.update_traces(textposition='outside')
fig.update_layout(height=400, showlegend=False)
fig.show()

## 8 – Als WeldVolumeModel speichern

In [ ]:
model = WeldVolumeModel(
    model_id=SCAN_FILE.stem,
    source_type='real',
    source_file=SCAN_FILE,
    point_cloud=pcd_final,
    preprocessing_report=report,
    metadata={
        'scanner': 'Hexagon CMM (PC-DMIS)',
        'scan_directions': len(unique_normals),
        'raw_points': len(pcd_raw.points),
    },
)

save_path = model.save(Path('../data/processed/cmm_scans'))
print(f'Gespeichert: {save_path}')
print(f'Modell:      {model}')

---
## 9 – Nächste Schritte

- [ ] Preprocessing-Parameter an reale Daten tunen (bes. `voxel_size` relativ zur KMM-Auflösung)
- [ ] Vergleich: KMM-Scan vs. CAD-Ideal (AP2.2 Subtraktionsmethode)
- [ ] Scanlinien-Normalen als Zusatzfeature für Segmentierung nutzen?
- [ ] Mehrere Proben vergleichen (Batch-Exploration)